In [ ]:
import os 
from getpass import getpass
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.core import PromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.chains import LLMChain
from IPython.display import display, Markdown
from langchain_core.runnables import RunnableLambda

from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel



load_dotenv()


ModuleNotFoundError: No module named 'langchain.core'

In [ ]:
# must enter API key 
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

# below should not be changed
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

# you can change this as preferred
os.environ["LANGCHAIN_PROJECT"] = "damar-ananta-langsmith-openai"

In [ ]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# For normal accurate responses
llm = ChatOpenAI(temperature=0.0, model="gpt-4o-mini")

# Tradition Chains vs LCEL

## Traditional LLMChain

In [ ]:
prompt_template = "Give me a small report on {topic}" #prompt_template merupakan template prompt yang memiliki placeholder:

prompt = PromptTemplate(
    input_variables=["topic"], #menjelaskan bahwa prompt ini membutuhkan input dengan nama variable topic.
    template=prompt_template
)

In [ ]:
llm_out = llm.invoke("Hello there") #invoke() digunakan untuk menjalankan komponen LangChain dengan input.
llm_out

In [ ]:
output_parser = StrOutputParser() #StrOutputParser bertugas mengubah output dari LLM menjadi string biasa.


In [ ]:
out = output_parser.invoke(llm_out)
out

In [ ]:
# Menggabungkan dengan LLMChain
chain = LLMChain(prompt=prompt, llm=llm, output_parser=output_parser) #Artinya, LLMChain tidak hanya menghasilkan teks, tetapi mengembalikan output dengan key tertentu.

In [ ]:
result = chain.invoke("retrieval augmented generation")

In [ ]:
display(Markdown(result["text"]))

## Retrieval Augmented Generation(RAG)

## Langhain Expression Language (LCEL)

In [ ]:
lcel_chain = prompt | llm | output_parser

In [ ]:
# Menjalankan LCEL Chain
result = lcel_chain.invoke("retrieval augmented generation")
result

In [ ]:
display(Markdown(result))

### Retrieval Augmented Generation

### How Does the Pipe Operator Work?

In [ ]:
class Runnable: #method ini membuat function baru yang menjalankan dua komponen secara berurutan.
    def __init__(self, func):
        self.func = func #Ketika object Runnable dibuat, function yang diberikan disimpan di dalam attribute
    def __or__(self, other): #merupakan special method Python yang digunakan ketika object menggunakan operator
        def chained_func(*args, **kwargs):
            return other.invoke(self.func(*args, **kwargs))
        return Runnable(chained_func)
    def invoke(self, *args, **kwargs): #Jadi invoke() adalah cara untuk menjalankan function tersebut melalui object Runnable.
        return self.func(*args, **kwargs)

In [ ]:
def add_five(x):
    return x+5

def sub_five(x):
    return x-5

def mul_five(x):
    return x*5

In [ ]:
add_five_runnable = Runnable(add_five)
sub_five_runnable = Runnable(sub_five)
mul_five_runnable = Runnable(mul_five)

In [ ]:

chain = (add_five_runnable).__or__(sub_five_runnable).__or__(mul_five_runnable)

chain.invoke(3)

In [ ]:
chain = add_five_runnable | sub_five_runnable | mul_five_runnable

chain.invoke(3)

## LCEL RunnableLambda

In [ ]:
add_five_runnable = RunnableLambda(add_five) #RunnableLambda merupakan class bawaan LangChain yang digunakan untuk mengubah function Python biasa menjadi Runnable.
sub_five_runnable = RunnableLambda(sub_five)
mul_five_runnable = RunnableLambda(mul_five)

In [ ]:
#Menggabungkan LLM dengan Function Python

chain = add_five_runnable | sub_five_runnable | mul_five_runnable

In [ ]:
chain.invoke(3)

In [ ]:
#Menggabungkan LLM dengan Function Python
prompt_str = "give me a small report about {topic}"
prompt = PromptTemplate(
    input_variables=["topic"],
    template=prompt_str
)

In [ ]:
chain = prompt | llm | output_parser

In [ ]:

result = chain.invoke("AI")
display(Markdown(result))

### Artificial Intelligence

In [ ]:
def extract_fact(x):
    if "\n\n" in x:
        return "\n".join(x.split("\n\n")[1:])
    else:
        return x

old_word = "AI"
new_word = "skynet"

def replace_word(x):
    return x.replace(old_word, new_word)

In [ ]:
extract_fact_runnable = RunnableLambda(extract_fact)
replace_word_runnable = RunnableLambda(replace_word)

In [ ]:
chain = prompt | llm | output_parser | extract_fact_runnable | replace_word_runnable

In [ ]:
result = chain.invoke("retrieval augmented generation")
display(Markdown(result))

## LCEL RunnableParallel and RunnablePassthrough

In [ ]:
embedding = OpenAIEmbeddings()

vecstore_a = DocArrayInMemorySearch.from_texts(
    [
        "half the info is here",
        "DeepSeek-V3 was released in December 2024"
    ],
    embedding=embedding
)
vecstore_b = DocArrayInMemorySearch.from_texts(
    [
        "the other half of the info is here",
        "the DeepSeek-V3 LLM is a mixture of experts model with 671B parameters"
    ],
    embedding=embedding
)

In [ ]:
prompt_str = """Using the context provided, answer the user's question.
Context:
{context_a}
{context_b}
"""

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(prompt_str),
    HumanMessagePromptTemplate.from_template("{question}")
])

In [ ]:
retriever_a = vecstore_a.as_retriever()
retriever_b = vecstore_b.as_retriever()

retrieval = RunnableParallel( #Digunakan untuk menjalankan beberapa Runnable secara paralel.
    {
        "context_a": retriever_a, "context_b": retriever_b, "question": RunnablePassthrough() #Digunakan untuk meneruskan input ke komponen berikutnya tanpa mengubahnya.
    }
)

In [ ]:

chain = retrieval | prompt | llm | output_parser

In [ ]:

result = chain.invoke(
    "what architecture does the model DeepSeek released in december use?"
)
result
